REALWEAR INDUSTRIAL OPERATIONS ANALYTICS
         
WELCOME TO STATISTICAL ANALYSIS

In [3]:
# Import the Required Libraries
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import(shapiro,levene,f_oneway,ttest_ind,pearsonr,
                        spearmanr,chi2_contingency)
import sqlite3
import warnings
warnings.filterwarnings('ignore')

print("All Libraries imported successfully!")

All Libraries imported successfully!


In [4]:
#Load data From Sqlite

conn=sqlite3.connect(r'C:\Users\BIT\OneDrive\Desktop\REAL WEAR PROJECT\sql\database\realwear.db')

#Load main table
df=pd.read_sql_query("Select * from master_session_log",conn)
df_workers=pd.read_sql_query("Select * from Worker_Master",conn)
df_devices=pd.read_sql_query("Select * from device_health_log",conn)

conn.close()


In [40]:
print(f"Master Session Log:{df.info()}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 30 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Session_ID             12000 non-null  object 
 1   Session_Date           12000 non-null  object 
 2   Shift                  12000 non-null  object 
 3   Plant_Location         12000 non-null  object 
 4   Plant_Zone             12000 non-null  object 
 5   Department             12000 non-null  object 
 6   Worker_ID              12000 non-null  object 
 7   Worker_Role            12000 non-null  object 
 8   Device_ID              12000 non-null  object 
 9   Platform               12000 non-null  object 
 10  Issue_Type             12000 non-null  object 
 11  Issue_Priority         12000 non-null  object 
 12  Meeting_Duration_min   12000 non-null  float64
 13  Resolution_Time_min    12000 non-null  float64
 14  Command_Attempts       12000 non-null  int64  
 15  Co

In [ ]:
# Descriptive Statistics
#numeric columns

numeric_cols=[]
categorical_cols=[]
date_cols=[]
for i in df.columns:
    if df[i].dtype!='object':
        numeric_cols.append(i)
    elif df[i].dtype=='datetime64[ns]':
        date_cols.append(i)
    else:
        categorical_cols.append(i)

print(f"NUMERIC COLUMNS ({len(numeric_cols)}):")
print(f"CATEGORICAL COLUMNS ({len(categorical_cols)}):")
print(f"DateTime COLUMNS ({len(date_cols)}):")




NUMERIC COLUMNS (12):
CATEGORICAL COLUMNS (18):
DateTime COLUMNS (0):


In [34]:
# Complete Descriptive statsicts

desc_stats=df[numeric_cols].describe().T  #Transpose
desc_stats['skewness']=df[numeric_cols].skew()
desc_stats['median']=df[numeric_cols].median()
desc_stats['variance']=df[numeric_cols].var()   #population variance
desc_stats['cv%']=(df[numeric_cols].std()/df[numeric_cols].mean()*100).round(2)

print("=" * 60)
print("DESCRIPTIVE STATISTICS SUMMARY")
print("=" * 60)
print(desc_stats.round(2))

DESCRIPTIVE STATISTICS SUMMARY
                         count   mean    std   min    25%    50%   75%    max  \
Meeting_Duration_min   12000.0  29.89  14.59   5.0  17.27  29.70  42.7   55.0   
Resolution_Time_min    12000.0  49.26  29.05   4.4  25.80  44.40  68.3  137.1   
Command_Attempts       12000.0  27.28  10.44  10.0  18.00  27.00  37.0   45.0   
Command_Failures       12000.0   5.60   4.58   0.0   2.00   4.00   8.0   24.0   
Command_Success_Rate   12000.0  79.91  13.63  40.0  68.40  83.30  90.7  100.0   
Noise_Level_dB         12000.0  81.39  20.99  45.0  63.30  81.50  99.4  118.0   
Signal_Strength_dBm    12000.0 -61.47  13.89 -85.0 -74.00 -62.00 -49.0  -38.0   
Battery_Start_%        12000.0  70.11  17.61  40.0  55.00  70.00  86.0  100.0   
Battery_End_%          12000.0  58.08  18.45  19.0  43.00  58.00  73.0   97.0   
battery_drain_percent  12000.0  12.47   5.51   3.0   7.70  12.45  17.2   22.0   
Downtime_Saved_min     12000.0  49.75  26.08   5.0  27.10  49.70  72.0   95.0 

Why We Need to check whether Datas are normal or not 
Reason-- Before Doing test we need to whether the data is applying or not because there are two types of test
Parametric and non-parametric.

Parametric tests are those test where we have information about population (like mean,variance)
For this data should be normal

Non-Parametric test are those where we dont need any information about population


In [39]:
# NORAMALITY TEST (SHAPIRO-WILK)

from scipy.stats import shapiro

print("="*40)
print("Normality Test Results")
print("="*40)
print(f"HO: Data is normally distributes")
print(f"H1: Data is not normally distributes")
print(f"Significance Level:0.05")

normality_results={}

for i in numeric_cols:
    ##Shapiro_Wilk works best with sample<5000
    # With 12,000 rows it becomes too sensitive — detects tiny deviations as non-normal
    sample=df[i].dropna().sample(5000,random_state=42)
    stat,p_value=shapiro(sample)

    normality_results[i]={'statistics':round(stat,4),
                          'pvalue':round(p_value,4),
                           'normal': 'Yes'if p_value>0.05 else 'No'}
    
    print(f"{i:30} | stat={stat:.4f} | p={p_value:.4f} | Normal: {'Yes' if p_value > 0.05 else 'No'}")


Normality Test Results
HO: Data is normally distributes
H1: Data is not normally distributes
Significance Level:0.05
Meeting_Duration_min           | stat=0.9540 | p=0.0000 | Normal: No
Resolution_Time_min            | stat=0.9526 | p=0.0000 | Normal: No
Command_Attempts               | stat=0.9502 | p=0.0000 | Normal: No
Command_Failures               | stat=0.8907 | p=0.0000 | Normal: No
Command_Success_Rate           | stat=0.9436 | p=0.0000 | Normal: No
Noise_Level_dB                 | stat=0.9567 | p=0.0000 | Normal: No
Signal_Strength_dBm            | stat=0.9524 | p=0.0000 | Normal: No
Battery_Start_%                | stat=0.9564 | p=0.0000 | Normal: No
Battery_End_%                  | stat=0.9762 | p=0.0000 | Normal: No
battery_drain_percent          | stat=0.9553 | p=0.0000 | Normal: No
Downtime_Saved_min             | stat=0.9544 | p=0.0000 | Normal: No
Productivity_Score             | stat=0.8463 | p=0.0000 | Normal: No


The Above result shows that data is not normally distributed so we have two option we can do
1. Tranforming the data using Log-transformation to normalise skewed data

2. We can Use non-parametric test like Mann_whitney Utest 
kruskal-Wallis test

Lets Explore Both the ways

In [57]:
# 1. Log Transformation

#Apply log transformation
#np.log1p will be used insted of np,log
# Reason would be 
# np.log(0)  = undefined (-infinity) 
#np.log1p(0) = log(1+0) = 0 
skewed_cols=numeric_cols.copy()
for col in skewed_cols:
    df[f'log_{col}']=np.log1p(df[col])
print(" log transformed")

for col in skewed_cols:
    print(f"log_{col}")

#Retest Normality on transformed Columns
log_cols=[f'log_{col}' for col in skewed_cols]

print("Normaality Test After Log transformation")

for col in log_cols:
     # Check if column exists and has enough values
    if col not in df.columns:
        print(f"{col:35} | Column not found")
        continue
    
    valid_data = df[col].dropna()
    
    if len(valid_data) == 0:
        print(f"{col:35} | No valid data")
        continue
    sample=df[col].dropna().sample(5000,random_state=42)
    stat,p_value=shapiro(sample)
    print(f"{col:35} | p={p_value:.4f} | Normal: {'Yes' if p_value > 0.05 else 'No'}")




 log transformed
log_Meeting_Duration_min
log_Resolution_Time_min
log_Command_Attempts
log_Command_Failures
log_Command_Success_Rate
log_Noise_Level_dB
log_Signal_Strength_dBm
log_Battery_Start_%
log_Battery_End_%
log_battery_drain_percent
log_Downtime_Saved_min
log_Productivity_Score
Normaality Test After Log transformation
log_Meeting_Duration_min            | p=0.0000 | Normal: No
log_Resolution_Time_min             | p=0.0000 | Normal: No
log_Command_Attempts                | p=0.0000 | Normal: No
log_Command_Failures                | p=0.0000 | Normal: No
log_Command_Success_Rate            | p=0.0000 | Normal: No
log_Noise_Level_dB                  | p=0.0000 | Normal: No
log_Signal_Strength_dBm             | No valid data
log_Battery_Start_%                 | p=0.0000 | Normal: No
log_Battery_End_%                   | p=0.0000 | Normal: No
log_battery_drain_percent           | p=0.0000 | Normal: No
log_Downtime_Saved_min              | p=0.0000 | Normal: No
log_Productivity_Scor


CONCLUSION:
- All numeric columns failed normality test before transformation
- Log transformation applied to skewed columns
- All columns still failed normality test after transformation
- Reason: Real world industrial data rarely follows perfect 
  normal distribution with 12,000 rows
- Decision: Use NON-PARAMETRIC tests for all hypothesis testing
  
  T-Test → Mann-Whitney U Test
  ANOVA  → Kruskal-Wallis Test
  
- Effect Size will be calculated alongside p-values to measure
  PRACTICAL significance not just statistical significance
  
  Data was not perfectly normal however given our large sample size of 12,000 rows Central Limit Theorem applies. I ran both parametric tests (ANOVA, T-Test) and non-parametric alternatives (Kruskal-Wallis, Mann-Whitney) and compared results for robustness.


In [68]:
# CENTRAL LIMIT THEOREM VERIFICATION
df['Noise_Level_dB'] = pd.to_numeric(df['Noise_Level_dB'], errors='coerce')
print(f'Total sample size:{len(df)}')
print()

groups={
    'Platform groups':df.groupby('Platform').size(),
    'Shift groups':df.groupby('Shift').size(),
    'Plant groups':df.groupby('Plant_Location').size(),
    'Noise_category Groups': df.groupby(pd.cut(df['Noise_Level_dB'],bins=[0,70,85,100],labels=['Low','medium','High'])).size(),
    'External Mic Groups':df.groupby('External_Mic_Used').size()


}
for group_name ,size in groups.items():
    print(f"{group_name}")
    for name ,size in size.items():
        clt_status="CLT APPLIES" if size>30 else "CLT CANNOT APPLY"
        print(f"{name}:{size} {clt_status}")

Total sample size:12000

Platform groups
MS Teams:8189 CLT APPLIES
Webex:3811 CLT APPLIES
Shift groups
Afternoon (14:00-22:00):4031 CLT APPLIES
Morning (06:00-14:00):3964 CLT APPLIES
Night (22:00-06:00):4005 CLT APPLIES
Plant groups
Bokaro Manufacturing Hub:2433 CLT APPLIES
Chennai Refinery Unit:2432 CLT APPLIES
Jamshedpur Steel Plant:2403 CLT APPLIES
Pune Automation Plant:2421 CLT APPLIES
Vadodara Chemical Unit:2311 CLT APPLIES
Noise_category Groups
Low:4150 CLT APPLIES
medium:2423 CLT APPLIES
High:2527 CLT APPLIES
External Mic Groups
No:6380 CLT APPLIES
Yes:5620 CLT APPLIES


Hypothesis Test 1 — Does Noise Level Affect Command Failures?

P-value --> Tells Is this Difference real or random 
 Means How likely we see the difference to this or more than this extreme if we assume Null hypothesis

 Effect Size-->It Tells Is id Diiference big enough to matter

Formula
η² = SS_between / SS_tota ,where

SS_between = variance BETWEEN groups
             (how different groups are from each other)

SS_total   = total variance in entire dataset
             (overall spread of all data)

η²         = proportion of total variance explained 
             by group membership


In [67]:
#using One way ANOVA
print("H0: Noise level has no significant effect on command failures")
print("H1: Noise level has significant effect on command failures")
print("Test: One Way ANOVA (CLT justified — all groups n > 30)")

#Create noise Category
df['Noise_Category']=pd.cut(df['Noise_Level_dB'],bins=[0,70,85,100],labels=['Low','Medium','High'])

low_noise = df[df['Noise_Category'] == 'Low']['Command_Failures'].dropna()
medium_noise = df[df['Noise_Category'] == 'Medium']['Command_Failures'].dropna()
high_noise = df[df['Noise_Category'] == 'High']['Command_Failures'].dropna()

#Group statistics

f_stat,p_value=f_oneway(low_noise,medium_noise,high_noise)
print(f"\nANOVA RESULTS:")
print(f"  F-Statistic : {f_stat:.4f}")
print(f"  P-Value     : {p_value:.6f}")

if p_value<0.05:
    print("Reject Null Hypothesis")
    print('Noise Level has significant effect on Command Failures')

else:
    print("Fail to reject Null Hypothesis")
    print("Noise Level has No significant effect on command failures")

# Now here we will calculate the effect size means Power 
all_groups=pd.concat([low_noise,medium_noise,high_noise])
grand_mean=all_groups.mean()

ss_between = (len(low_noise) * (low_noise.mean() - grand_mean)**2 +
              len(medium_noise) * (medium_noise.mean() - grand_mean)**2 +
              len(high_noise) * (high_noise.mean() - grand_mean)**2)

ss_total = ((all_groups - grand_mean)**2).sum()
eta_squared = ss_between / ss_total

print(f"\nEFFECT SIZE (Eta Squared): {eta_squared:.4f}")
print(f"Interpretation: ", end="")
if eta_squared < 0.01:
    print("Negligible effect")
elif eta_squared < 0.06:
    print("Small effect")
elif eta_squared < 0.14:
    print("Medium effect")
else:
    print("Large effect")


H0: Noise level has no significant effect on command failures
H1: Noise level has significant effect on command failures
Test: One Way ANOVA (CLT justified — all groups n > 30)

ANOVA RESULTS:
  F-Statistic : 153.3946
  P-Value     : 0.000000
Reject Null Hypothesis
Noise Level has significant effect on Command Failures

EFFECT SIZE (Eta Squared): 0.0326
Interpretation: Small effect


In [ ]:
#